# 🐍 Part 2: Database Enrichment & Data Wrangling via Python
## Refactoring and Enriching the `Owners` Table (`03_update_owners.ipynb`)


## 🎯 Context & Objectives
The goal of this stage is to perform strategic data enrichment on the `owners` table within the `insight_places` database using Python (`03_update_owners.ipynb`). By renaming legacy contact fields (`email`), adding structured contact details (`phone_number`), enforcing temporal creation rules prior to property creation (`host_since`), and `redistributing property ownership` to reflect realistic market dynamics, I will turn this table into a robust owner dimension for advanced analytical queries and BI dashboards.
This notebook represents the third module of a sequential ETL execution, preserving complete relational integrity with the `listings` table.



## 🛠️ Environment Setup & Dependencies

To execute this ETL pipeline securely and efficiently, the following Python libraries are required: 
* **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing DDL/DML operations. 
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys. 
* **`pandas`**: Data manipulation, aggregation, and structural validation. 
* **`random` & `numpy`**: Native and mathematical modules used to generate realistic pricing distributions and conditional business logic. 
* **`faker`**: Synthetic data generation library used to produce realistic English names and identity metadata (en_US locale).


## 📌 Refining the `Owners` Table 

This module focuses on upgrading owner profiles, contact metadata, and relational integrity with properties:

1. **Schema Modernization & Contact Enrichment**: Renaming the legacy `contact` field to `email`, updating domain names **(@gmail.com, @yahoo.com, @outlook.com)**, and adding a formatted `phone_number` column.

2. **Name Standardization (`owner_name`)**: Replacing legacy localized names with realistic, fully **English full names** using the Faker library.

3. **Property Redistribution (`owner_id in listings`)**: Recalibrating the one-to-many relationship using a `Pareto/Power Law distribution` **(~70% single property, ~20% 2-3 properties, ~10% 4-10 properties)**.

4. **Aggregated Inventory Synchronization (`total_listings`)**: Dynamically updating the total property count for each owner based on the newly assigned listings.

5. **Temporal Integrity Enforcer (`host_since`)**: Generating platform registration dates that strictly precede or coincide with the earliest creation timestamp (`created_at`) of their owned properties in listings.

6. **Performance Badge Setup (`is_superhost`)**: Adding and initializing the high-performer boolean flag for future calibration with booking performance metrics.

### 1) Installing and importing the libraries
In this step, I will load the libraries that will be used throughout the project.


In [1]:
!pip install sqlalchemy
!pip install pymysql
!pip install faker

import pandas as pd
import random
import datetime
import pymysql
from faker import Faker
from sqlalchemy import create_engine, text

fake = Faker('en_US')

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------ --------------------------------- 0.3/2.1 MB 6.7 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 16.4 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 16.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 14.7 MB/s eta 0:00:00


#### 1.1) Installing the data protection library
I will install the python-dotenv library to ensure the security of the MySQL password.


In [2]:
!pip install python-dotenv

#### 1.2) Environment Setup and Security Validation

In [3]:
# Installing and Importing libraries
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print(" System ready for database connection.") 
else: print("❌ Error: One or more credentials are missing from the .env file.")


🔒 Environment variables loaded successfully.
 System ready for database connection.


#### 1.3) Database Engine Initialization
Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.


In [4]:
# Creating the database connection engine using validated credentials

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")


🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Owners` Table

This section executes the step-by-step transformation pipeline to modernize contact details, apply realistic property distribution logic, and ensure strict temporal integrity.


### 2.1 Schema Modernization (DDL Adjustments)

In this step, I modify the `owners` table structure by renaming the legacy `contact` column to `email` and adding missing attributes (`phone_number`, `host_since`, `is_superhost`) to support enriched analytical queries.


In [5]:
def update_owners_schema():
    with engine.begin() as conn:
        print("Checking existing columns in 'owners' table...")
        
        # Inspection of current columns
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'owners'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # 1. Rename 'contact' to 'email' if legacy column exists
        if 'contact' in existing_columns and 'email' not in existing_columns:
            conn.execute(text("ALTER TABLE owners CHANGE COLUMN contact email VARCHAR(255);"))
            print("  `contact` column successfully renamed to `email`.")
        
        # 2. Add 'phone_number' if missing
        if 'phone_number' not in existing_columns:
            conn.execute(text("ALTER TABLE owners ADD COLUMN phone_number VARCHAR(30);"))
            print("  `phone_number` column successfully added.")
            
        # 3. Add 'host_since' if missing
        if 'host_since' not in existing_columns:
            conn.execute(text("ALTER TABLE owners ADD COLUMN host_since DATETIME;"))
            print("  `host_since` column successfully added.")
            
        # 4. Add 'is_superhost' if missing
        if 'is_superhost' not in existing_columns:
            conn.execute(text("ALTER TABLE owners ADD COLUMN is_superhost TINYINT DEFAULT 0;"))
            print("  `is_superhost` column successfully added.")

        print("✅ Schema modernization completed successfully!")

update_owners_schema()

Checking existing columns in 'owners' table...
  `contact` column successfully renamed to `email`.
  `phone_number` column successfully added.
  `host_since` column successfully added.
  `is_superhost` column successfully added.
✅ Schema modernization completed successfully!


### 2.2 Name Standardization and Contact Enrichment (`owner_name`, `email`, `phone_number`)

Generating realistic English full names using the `Faker` library, constructing valid email addresses with primary domains (`@gmail.com`, `@yahoo.com`, `@outlook.com`), and creating formatted phone numbers.

In [6]:
def update_owner_contacts():
    with engine.begin() as conn:
        print("Fetching owner records to update contact metadata...")
        owners = conn.execute(text("SELECT owner_id FROM owners")).fetchall()
        
        domains = ['gmail.com', 'yahoo.com', 'outlook.com']
        updates = []
        
        for (owner_id,) in owners:
            # Generate English full name
            full_name = fake.name()
            
            # Sanitize name to create a clean email handle
            clean_handle = full_name.lower().replace(" ", ".").replace("'", "")
            domain = random.choice(domains)
            email = f"{clean_handle}@{domain}"
            
            # Generate formatted phone number: +1 (555) XXX-XXXX
            phone = f"+1 (555) {random.randint(100, 999)}-{random.randint(1000, 9999)}"
            
            updates.append({
                "owner_id": owner_id,
                "owner_name": full_name,
                "email": email,
                "phone_number": phone
            })
            
        update_query = text("""
            UPDATE owners 
            SET owner_name = :owner_name,
                email = :email,
                phone_number = :phone_number
            WHERE owner_id = :owner_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Owner names, emails, and phone numbers successfully updated!")

update_owner_contacts()

Fetching owner records to update contact metadata...
✅ Owner names, emails, and phone numbers successfully updated!


### 2.3 Spatial & Portfolio Redistribution (`owner_id` in `listings`)

Reallocating property ownership across the `listings` table using a Pareto/Power Law distribution. This allocates single properties to ~70% of owners, 2-3 properties to ~20%, and multi-property portfolios (4-10 units) to ~10% of high-capacity hosts.


In [7]:
def redistribute_listing_ownership():
    with engine.begin() as conn:
        print("Fetching owners and listings to apply portfolio distribution...")
        
        # 1. Fetch all available owner IDs and listing IDs
        owner_ids = [row[0] for row in conn.execute(text("SELECT owner_id FROM owners")).fetchall()]
        listing_ids = [row[0] for row in conn.execute(text("SELECT listing_id FROM listings")).fetchall()]
        
        if not owner_ids or not listing_ids:
            print("❌ Error: Missing owners or listings records.")
            return

        # Shuffle to ensure unbiased assignment
        random.shuffle(owner_ids)
        random.shuffle(listing_ids)
        
        total_owners = len(owner_ids)
        
        # 2. Segment owners according to portfolio capacity tiers
        tier_1_count = int(total_owners * 0.70)  # 1 property
        tier_2_count = int(total_owners * 0.20)  # 2 to 3 properties
        # Remaining owners go to Tier 3 (4 to 10 properties)
        
        tier_1_owners = owner_ids[:tier_1_count]
        tier_2_owners = owner_ids[tier_1_count : tier_1_count + tier_2_count]
        tier_3_owners = owner_ids[tier_1_count + tier_2_count :]
        
        owner_listing_assignments = []
        unassigned_listings = listing_ids.copy()
        
        # Assign Tier 1 (1 property each)
        for owner in tier_1_owners:
            if unassigned_listings:
                owner_listing_assignments.append((owner, unassigned_listings.pop()))
                
        # Assign Tier 2 (2 to 3 properties each)
        for owner in tier_2_owners:
            num_properties = random.randint(2, 3)
            for _ in range(num_properties):
                if unassigned_listings:
                    owner_listing_assignments.append((owner, unassigned_listings.pop()))
                    
        # Assign Tier 3 (4 to 10 properties each)
        for owner in tier_3_owners:
            num_properties = random.randint(4, 10)
            for _ in range(num_properties):
                if unassigned_listings:
                    owner_listing_assignments.append((owner, unassigned_listings.pop()))
                    
        # Allocate any remaining listings randomly among Tier 2 and Tier 3 owners
        active_investors = tier_2_owners + tier_3_owners if (tier_2_owners + tier_3_owners) else owner_ids
        while unassigned_listings:
            selected_owner = random.choice(active_investors)
            owner_listing_assignments.append((selected_owner, unassigned_listings.pop()))
            
        # 3. Update the listings table with new owner assignments
        updates = [{"owner_id": owner, "listing_id": listing} for owner, listing in owner_listing_assignments]
        
        update_query = text("""
            UPDATE listings 
            SET owner_id = :owner_id 
            WHERE listing_id = :listing_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ Property ownership successfully redistributed across the `listings` table!")

redistribute_listing_ownership()

Fetching owners and listings to apply portfolio distribution...
✅ Property ownership successfully redistributed across the `listings` table!


### 2.4 Aggregated Inventory Synchronization (`total_listings`)

Calculating the exact count of properties assigned to each owner in the `listings` table and updating the `total_listings` field in the `owners` table to guarantee relational integrity.

In [8]:
def sync_total_listings():
    with engine.begin() as conn:
        print("Calculating property counts per owner...")
        
        # 1. Reset all total_listings to 0 to start fresh
        conn.execute(text("UPDATE owners SET total_listings = 0"))
        
        # 2. Count listings per owner
        count_query = text("""
            SELECT owner_id, COUNT(*) as property_count 
            FROM listings 
            GROUP BY owner_id
        """)
        
        owner_counts = conn.execute(count_query).fetchall()
        
        # 3. Update the owners table with the accurate counts
        updates = [
            {"owner_id": owner_id, "total_listings": count} 
            for owner_id, count in owner_counts
        ]
        
        if updates:
            update_query = text("""
                UPDATE owners 
                SET total_listings = :total_listings 
                WHERE owner_id = :owner_id
            """)
            conn.execute(update_query, updates)
            
        print("✅ `total_listings` successfully synchronized with the `listings` table!")

sync_total_listings()

# Verification query
def check_total_listings_distribution():
    with engine.connect() as conn:
        df = pd.read_sql(text("""
            SELECT total_listings, COUNT(*) as number_of_owners 
            FROM owners 
            GROUP BY total_listings 
            ORDER BY total_listings
        """), conn)
        print("\n📊 Property Distribution Check:")
        print(df)

check_total_listings_distribution()

Calculating property counts per owner...
✅ `total_listings` successfully synchronized with the `listings` table!

📊 Property Distribution Check:
   total_listings  number_of_owners
0               0              1807
1               1              6988
2               2               571
3               3               618


### 2.5 Temporal Integrity Enforcer (`host_since`)

Generating platform registration timestamps (`host_since`) that strictly precede or coincide with the earliest property creation date (`created_at` from `listings`) associated with each owner. For hosts with zero active listings, a randomized baseline registration date is assigned.

In [9]:
def update_host_since_dates():
    with engine.begin() as conn:
        print("Enforcing temporal integrity between owners and listings...")
        
        # 1. Fetch earliest listing creation date for owners with properties
        query_with_listings = text("""
            SELECT l.owner_id, MIN(l.created_at) as earliest_listing
            FROM listings l
            GROUP BY l.owner_id
        """)
        owners_with_listings = conn.execute(query_with_listings).fetchall()
        
        updates = []
        
        for owner_id, earliest_listing in owners_with_listings:
            if earliest_listing:
                # Subtract between 30 and 730 days (up to 2 years before property creation)
                days_prior = random.randint(30, 730)
                host_since = earliest_listing - datetime.timedelta(days=days_prior)
                updates.append({"owner_id": owner_id, "host_since": host_since})
                
        # 2. Assign default registration dates for owners with zero listings
        query_without_listings = text("""
            SELECT owner_id 
            FROM owners 
            WHERE total_listings = 0
        """)
        owners_without_listings = conn.execute(query_without_listings).fetchall()
        
        # Baseline platform launch window (e.g., 2020 to 2022)
        base_start = datetime.datetime(2020, 1, 1)
        base_end = datetime.datetime(2022, 12, 31)
        time_delta_seconds = int((base_end - base_start).total_seconds())
        
        for (owner_id,) in owners_without_listings:
            random_seconds = random.randint(0, time_delta_seconds)
            host_since = base_start + datetime.timedelta(seconds=random_seconds)
            updates.append({"owner_id": owner_id, "host_since": host_since})
            
        # 3. Batch update host_since in owners table
        update_query = text("""
            UPDATE owners 
            SET host_since = :host_since 
            WHERE owner_id = :owner_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ `host_since` timestamps successfully updated with temporal integrity!")

update_host_since_dates()

Enforcing temporal integrity between owners and listings...
✅ `host_since` timestamps successfully updated with temporal integrity!


### 2.6 Performance Badge Setup (`is_superhost`)

Assigning an initial ~18% distribution for the `is_superhost` status across property owners to simulate top-performing host recognition, pending final validation against historical booking and review performance metrics.

In [10]:
def update_superhost_status():
    with engine.begin() as conn:
        print("Calibrating `is_superhost` status across owners...")
        
        # 1. Fetch all owner IDs
        owners = conn.execute(text("SELECT owner_id FROM owners")).fetchall()
        
        # 2. Assign ~18% superhost probability
        updates = []
        for (owner_id,) in owners:
            is_superhost = 1 if random.random() < 0.18 else 0
            updates.append({"owner_id": owner_id, "is_superhost": is_superhost})
            
        # 3. Batch update owners table
        update_query = text("""
            UPDATE owners 
            SET is_superhost = :is_superhost 
            WHERE owner_id = :owner_id
        """)
        
        conn.execute(update_query, updates)
        print("✅ `is_superhost` flag successfully initialized!")

update_superhost_status()

# Final Verification
def verify_owners_refactoring():
    with engine.connect() as conn:
        df = pd.read_sql(text("SELECT * FROM owners LIMIT 10"), conn)
        print("\n🔍 Sample Refactored Owners Data:")
        print(df)

verify_owners_refactoring()

Calibrating `is_superhost` status across owners...
✅ `is_superhost` flag successfully initialized!

🔍 Sample Refactored Owners Data:
  owner_id         owner_name          tax_id                        email  \
0        1     Christina Ross  769.328.415-09     christina.ross@yahoo.com   
1       10  Benjamin Bartlett  301.729.654-70  benjamin.bartlett@yahoo.com   
2      100         Morgan Lee  894.361.520-51         morgan.lee@yahoo.com   
3     1000     Nicole Ramirez  962.704.351-61     nicole.ramirez@gmail.com   
4    10000     Madeline Brown  927.140.583-88   madeline.brown@outlook.com   
5     1001        Kayla Welch  407.956.381-75      kayla.welch@outlook.com   
6     1002    Jennifer Jordan  816.273.059-12  jennifer.jordan@outlook.com   
7     1003        Shawn Reese  136.807.542-80        shawn.reese@yahoo.com   
8     1004       Michael Cook  213.645.807-80       michael.cook@gmail.com   
9     1005          Blake Lee  485.763.190-39          blake.lee@gmail.com   

   total